# 17 CNN-MNIST 模型结构拆解

上一节已经知道 CNN-MNIST 要做什么：

```text
输入手写数字图片，输出 0 到 9 的类别预测。
```

这一节继续往实战靠近，但先不写代码。

我们只做一件事：

```text
把 CNN-MNIST 的模型结构逐层拆开，看每一层输入输出形状怎么变化。
```

这一节非常重要。

因为后面真正写 CNN 代码时，最容易出错的地方通常不是卷积层本身，而是：

```text
Flatten 后接全连接层时，维度算错。
```

## 1. 这一节为什么先拆模型结构

CNN 实战里，代码只是外壳。

真正要理解的是数据怎么流过网络。

如果你能在写代码前说清楚：

```text
输入是什么形状？
经过第一层卷积后变成什么形状？
经过池化后变成什么形状？
Flatten 后是多少维？
最后为什么输出 10 个分数？
```

那后面写代码就会稳很多。

所以这一节先把模型结构算清楚。

## 2. 先确定输入形状

MNIST 是灰度图。

单张图片可以理解成：

```text
1 x 28 x 28
```

如果按 batch 训练，假设 batch size 是 B，那么输入形状是：

$$
B \times 1 \times 28 \times 28
$$

其中：

- $B$ 表示这一批有多少张图片。
- $1$ 表示灰度图只有 1 个通道。
- $28$ 表示高度。
- $28$ 表示宽度。

后面我们只跟踪形状变化，先不关心具体 batch size 是多少。

## 3. 本节使用一个入门 CNN 结构

为了方便理解，我们使用一个很经典的入门结构，思想接近 LeNet：

```text
输入图片
-> 卷积层 1
-> ReLU
-> 池化层 1
-> 卷积层 2
-> ReLU
-> 池化层 2
-> Flatten
-> 全连接层 1
-> ReLU
-> 全连接层 2
-> ReLU
-> 全连接层 3
-> 输出 10 个类别分数
```

注意：这里不是说所有 CNN-MNIST 都必须这么写。

它只是一个适合初学者理解的结构。

## 4. 第一层卷积：从 1 张图变成 6 张特征图

第一层卷积可以这样设想：

```text
输入通道数：1
输出通道数：6
卷积核大小：5 x 5
stride：1
padding：0
```

输入形状是：

```text
B x 1 x 28 x 28
```

输出通道数是 6，表示这一层有 6 个卷积核，会输出 6 张特征图。

高度和宽度按卷积输出公式计算：

$$
\text{输出尺寸}=\frac{\text{输入尺寸}-\text{卷积核尺寸}+2\times\text{padding}}{\text{stride}}+1
$$

代入高度或宽度：

$$
\frac{28-5+2\times0}{1}+1=24
$$

所以第一层卷积后形状是：

```text
B x 6 x 24 x 24
```

## 5. ReLU 后形状不变

卷积层后面通常会接 ReLU。

ReLU 的作用是处理数值，让模型有更强的表达能力。

但 ReLU 通常不改变形状。

所以：

```text
卷积后：B x 6 x 24 x 24
ReLU 后：B x 6 x 24 x 24
```

可以这样记：

```text
ReLU 改数值，不改形状。
```

## 6. 第一次池化：高和宽减半

接下来使用一个常见的最大池化：

```text
池化窗口：2 x 2
stride：2
```

池化通常不改变通道数。

它主要压缩高度和宽度。

输入是：

```text
B x 6 x 24 x 24
```

经过 2 x 2、stride 为 2 的池化后，高和宽从 24 变成 12。

所以输出形状是：

```text
B x 6 x 12 x 12
```

这里要注意：

```text
通道数还是 6。
变小的是每张特征图的高和宽。
```

## 7. 第二层卷积：从 6 张特征图变成 16 张特征图

第二层卷积接收的是上一层池化后的特征图：

```text
B x 6 x 12 x 12
```

这一层可以设想为：

```text
输入通道数：6
输出通道数：16
卷积核大小：5 x 5
stride：1
padding：0
```

输出通道数是 16，表示这一层会输出 16 张特征图。

高度和宽度继续套公式：

$$
\frac{12-5+2\times0}{1}+1=8
$$

所以第二层卷积后形状是：

```text
B x 16 x 8 x 8
```

这一步要特别注意：

```text
卷积层的输出通道数由卷积核个数决定。
```

## 8. 第二次池化：再次压缩特征图

第二层卷积后接 ReLU，形状不变：

```text
B x 16 x 8 x 8
```

然后再做一次 2 x 2、stride 为 2 的最大池化。

高度和宽度从 8 变成 4。

所以池化后形状是：

```text
B x 16 x 4 x 4
```

到这里，CNN 前半部分已经把原来的图片变成了一组更小、更抽象的特征图。

## 9. Flatten：把特征图展平成一维向量

全连接层不能直接接收 `B x 16 x 4 x 4` 这种四维形状。

所以需要 Flatten。

Flatten 会把每张图片的特征图展平成一维向量：

$$
16 \times 4 \times 4 = 256
$$

所以：

```text
B x 16 x 4 x 4
-> B x 256
```

这就是为什么后面的第一个全连接层，输入特征数要对应 256。

这里是 CNN 初学者最容易算错的地方。

## 10. 全连接层：从特征向量到类别分数

Flatten 后，每张图片得到一个 256 维特征向量。

接下来交给全连接层分类。

可以先设想：

```text
B x 256
-> B x 120
-> B x 84
-> B x 10
```

前两个全连接层可以理解成继续组合特征。

最后一层输出 10 个数，因为 MNIST 有 10 个类别。

所以最终输出形状是：

```text
B x 10
```

## 11. 为什么最后不是输出一个数字

这是一个常见误会。

MNIST 虽然最后预测的是一个数字，比如 7。

但模型通常不是直接输出一个“7”。

它输出的是 10 个类别分数：

```text
类别 0 的分数
类别 1 的分数
类别 2 的分数
...
类别 9 的分数
```

然后选择分数最高的类别作为预测结果。

所以：

```text
模型输出：10 个分数
最终预测：分数最高的那个类别
```

## 12. 把整个形状变化串起来

现在把整条路线串起来：

```text
输入图片：B x 1 x 28 x 28

卷积层 1：B x 6 x 24 x 24
ReLU：B x 6 x 24 x 24
池化层 1：B x 6 x 12 x 12

卷积层 2：B x 16 x 8 x 8
ReLU：B x 16 x 8 x 8
池化层 2：B x 16 x 4 x 4

Flatten：B x 256
全连接层 1：B x 120
全连接层 2：B x 84
全连接层 3：B x 10
```

这条形状变化路线，比代码本身更重要。

后面写模型时，代码只是把这条路线表达出来。

## 13. 哪些层有可学习参数

这个模型里，有可学习参数的层主要是：

```text
卷积层 1
卷积层 2
全连接层 1
全连接层 2
全连接层 3
```

它们里面有权重和 bias。

训练时，优化器会更新这些参数。

通常没有可学习参数的部分包括：

```text
ReLU
池化层
Flatten
```

它们虽然没有参数，但仍然参与前向传播和反向传播。

## 14. 这个结构和 LeNet 的关系

这一节使用的结构，和 LeNet 的思想很接近。

LeNet 的核心路线也是：

```text
卷积
-> 池化
-> 卷积
-> 池化
-> 全连接
-> 分类输出
```

这也是为什么 LeNet 很适合作为 CNN 入门网络。

它足够简单，但又完整包含 CNN 图像分类的核心流程。

## 15. 后面写代码时重点看什么

后面真正写代码时，不要先盯着代码语法。

要带着这几个问题看：

```text
第一层卷积为什么输入通道数是 1？
第一层卷积为什么输出通道数可以是 6？
池化为什么不改变通道数？
Flatten 为什么得到 256？
最后一层为什么输出 10？
```

如果这些问题都能回答，说明模型结构已经基本理解了。

## 16. 本节小结

这一节要记住：

```text
CNN-MNIST 模型不是一堆随机堆起来的层。
它是一条清楚的数据形状变化路线。
```

核心路线是：

```text
B x 1 x 28 x 28
-> B x 6 x 24 x 24
-> B x 6 x 12 x 12
-> B x 16 x 8 x 8
-> B x 16 x 4 x 4
-> B x 256
-> B x 10
```

其中：

```text
卷积层负责提取特征。
ReLU 负责增强表达能力。
池化层负责压缩特征图。
Flatten 负责展平。
全连接层负责分类。
```

## 17. 自测问题

1. MNIST 输入 CNN 前的单张图片形状是什么？
2. 如果 batch size 是 B，输入形状是什么？
3. 第一层卷积输入通道数为什么是 1？
4. 第一层卷积输出通道数为 6，表示什么？
5. 经过 2 x 2、stride 为 2 的池化，通道数会变吗？
6. 为什么 `B x 16 x 4 x 4` Flatten 后会变成 `B x 256`？
7. 最后一层为什么输出 10 个数？
8. 哪些层通常有可学习参数？
9. 哪些层通常没有可学习参数？
10. 为什么说形状变化路线比背代码更重要？